# Project 03 — Customer Churn Prediction

**Author:** Jorgo Luka  
**Format:** Standalone Google Colab notebook  
**Dataset:** UCI Iranian Churn — 3,150 telecom customers observed over 12 months  
**Core tools:** Python · Pandas · scikit-learn · Matplotlib · Seaborn

## Recruiter summary

This project builds a decision-ready churn screening system rather than an accuracy-only classifier. It protects the holdout set, isolates duplicate customer profiles between splits, compares a prevalence baseline with interpretable and nonlinear models, calibrates probabilities from grouped out-of-fold predictions and selects an intervention threshold using explicit decision costs.

| Capability | Evidence |
|---|---|
| Problem design | Nine-month feature window, three-month planning gap and end-of-year churn outcome |
| Leakage control | Duplicate-profile grouping, untouched holdout, age exclusion and `Status` proxy sensitivity test |
| Modelling | Dummy baseline, logistic regression, random forest and histogram gradient boosting |
| Evaluation | PR-AUC, ROC-AUC, Brier score, calibration, bootstrap intervals and failure slices |
| Decision layer | Cost-based threshold, capacity/alert rate, subgroup review and validated inference function |
| Engineering | Data/model cards, serialized model bundle, drift reference, tests and SHA-256 manifest |

> This is a screening model for prioritising retention review. It does not prove why a customer churned and must not trigger adverse treatment automatically.

## Business question and decision

**Question:** Which customers should a retention team review during a three-month planning window?

**Decision:** Flag customers when their calibrated churn probability exceeds a threshold chosen on training-only out-of-fold predictions.

**Primary model-selection metric:** average precision (PR-AUC), because only about one customer in six churns and accuracy can hide missed churners.

**Decision-cost assumptions:**

- Contacting or reviewing a flagged customer: 25 cost units.
- Missing a customer who later churns: 200 cost units.
- Correctly leaving a non-churner unflagged: 0 cost units.

These are transparent scenario weights, not claimed company currency or realised savings. A production owner should replace them with measured intervention costs and incremental retention value from an experiment.

## Data source, chronology and licence

- Dataset: [Iranian Churn, UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/563/iranian+churn+dataset)
- DOI: [10.24432/C5JW3Z](https://doi.org/10.24432/C5JW3Z)
- Licence: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

UCI states that the records were randomly collected from an Iranian telecom company's database over 12 months. Predictors summarize the first nine months; churn is measured at the end of month 12, leaving a three-month planning gap. The file contains no customer identifier or event timestamps, so the notebook uses grouped stratified validation rather than pretending it can perform a true calendar-time split.

## 0. Environment and reproducibility

Run **Runtime → Restart session and run all** in Google Colab. No private account or API key is required.

In [ ]:
%pip -q install "scikit-learn>=1.5,<1.9" "joblib>=1.3" "gradio>=5,<7"

In [ ]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import math
import platform
import random
import re
import urllib.request
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from scipy.special import expit, logit
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


@dataclass(frozen=True)
class ProjectConfig:
    dataset_url: str = "https://archive.ics.uci.edu/static/public/563/iranian+churn+dataset.zip"
    archive_name: str = "iranian_churn_dataset.zip"
    csv_name: str = "Customer Churn.csv"
    artifact_dir: str = "customer_churn_artifacts"
    holdout_folds: int = 5
    cross_validation_folds: int = 5
    contact_cost_units: float = 25.0
    missed_churn_cost_units: float = 200.0
    bootstrap_repetitions: int = 500
    permutation_repetitions: int = 20
    launch_app: bool = False


CFG = ProjectConfig()
DATA_DIR = Path("churn_source_data")
ARTIFACTS = Path(CFG.artifact_dir)
DATA_DIR.mkdir(exist_ok=True)
ARTIFACTS.mkdir(exist_ok=True)

print({
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "configuration": asdict(CFG),
})

## 1. Download and fingerprint the source

The notebook downloads the UCI archive directly, verifies its expected contents and records a SHA-256 fingerprint.

In [ ]:
REFERENCE_ARCHIVE_SHA256 = "696c3a1812267980751f30d19193a1b430ac4ad76172bb089e664c96813ead66"
ARCHIVE_PATH = DATA_DIR / CFG.archive_name
CSV_PATH = DATA_DIR / CFG.csv_name


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


if not ARCHIVE_PATH.exists():
    request = urllib.request.Request(CFG.dataset_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120) as response, ARCHIVE_PATH.open("wb") as target:
        while chunk := response.read(1 << 20):
            target.write(chunk)

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    assert CFG.csv_name in archive.namelist(), f"Expected {CFG.csv_name} in source archive"
    archive.extract(CFG.csv_name, DATA_DIR)

SOURCE_FINGERPRINT = {
    "dataset": "UCI Iranian Churn",
    "source_url": CFG.dataset_url,
    "doi": "10.24432/C5JW3Z",
    "archive_bytes": ARCHIVE_PATH.stat().st_size,
    "archive_sha256": sha256_file(ARCHIVE_PATH),
    "matches_reference_archive": sha256_file(ARCHIVE_PATH) == REFERENCE_ARCHIVE_SHA256,
    "retrieved_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
}
assert CSV_PATH.exists()
print(json.dumps(SOURCE_FINGERPRINT, indent=2))

## 2. Load, normalise and validate the data contract

Column names contain inconsistent double spaces in the source. Names are normalised deterministically; values are not edited or imputed because the pinned file has no missing entries.

In [ ]:
def normalise_column(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.strip().lower()).strip("_")


raw = pd.read_csv(CSV_PATH)
raw.columns = [normalise_column(column) for column in raw.columns]

EXPECTED_COLUMNS = [
    "call_failure",
    "complains",
    "subscription_length",
    "charge_amount",
    "seconds_of_use",
    "frequency_of_use",
    "frequency_of_sms",
    "distinct_called_numbers",
    "age_group",
    "tariff_plan",
    "status",
    "age",
    "customer_value",
    "churn",
]

assert raw.columns.tolist() == EXPECTED_COLUMNS

DATA_CONTRACT = pd.DataFrame([
    {"field": "churn", "role": "target", "allowed": "0 or 1", "timing": "end of month 12"},
    {"field": "complains", "role": "binary feature", "allowed": "0 or 1", "timing": "first 9 months"},
    {"field": "tariff_plan", "role": "service feature", "allowed": "1 or 2", "timing": "first 9 months"},
    {"field": "status", "role": "proxy-risk feature", "allowed": "1 or 2", "timing": "first 9 months"},
    {"field": "age_group", "role": "audit only", "allowed": "1 through 5", "timing": "customer attribute"},
    {"field": "age", "role": "audit only", "allowed": "non-negative", "timing": "customer attribute"},
    {"field": "remaining numeric fields", "role": "model features", "allowed": "non-negative", "timing": "first 9 months"},
])

quality_profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique": raw.nunique(),
    "minimum": raw.min(numeric_only=True),
    "maximum": raw.max(numeric_only=True),
}).reset_index(names="field")

assert raw.shape == (3150, 14)
assert raw.isna().sum().sum() == 0
assert set(raw["churn"].unique()) == {0, 1}
assert set(raw["complains"].unique()).issubset({0, 1})
assert set(raw["tariff_plan"].unique()).issubset({1, 2})
assert set(raw["status"].unique()).issubset({1, 2})
assert set(raw["age_group"].unique()).issubset({1, 2, 3, 4, 5})
assert (raw.drop(columns="churn") >= 0).all().all()

display(DATA_CONTRACT)
display(quality_profile)
print(f"Raw shape: {raw.shape[0]:,} customers × {raw.shape[1]} columns")

## 3. Duplicate-profile audit and leakage-safe grouping

The file has no customer ID. Exact duplicate rows could be repeated customers, different customers with the same aggregate profile or source duplication. Deleting them would invent an unsupported assumption. Randomly splitting them could place identical profiles in training and testing, making generalisation look better than it is.

This notebook therefore keeps every row but assigns identical predictor profiles to the same validation group. Even profiles with conflicting churn outcomes remain together.

In [ ]:
X_raw = raw.drop(columns="churn").copy()
y = raw["churn"].astype(int).copy()

OPERATIONAL_INPUT_COLUMNS = [
    "call_failure",
    "complains",
    "subscription_length",
    "charge_amount",
    "seconds_of_use",
    "frequency_of_use",
    "frequency_of_sms",
    "distinct_called_numbers",
    "tariff_plan",
    "customer_value",
]
operational_profiles = X_raw[OPERATIONAL_INPUT_COLUMNS]

full_duplicate_rows = int(raw.duplicated().sum())
feature_duplicate_rows = int(operational_profiles.duplicated().sum())
profile_groups = pd.util.hash_pandas_object(operational_profiles, index=False).astype(str)

profile_audit = (
    raw.groupby(OPERATIONAL_INPUT_COLUMNS, dropna=False)["churn"]
    .agg(rows="size", target_classes="nunique", churn_rate="mean")
    .reset_index()
)
mixed_label_profiles = int((profile_audit["target_classes"] > 1).sum())
rows_in_mixed_profiles = int(profile_audit.loc[profile_audit["target_classes"] > 1, "rows"].sum())

DUPLICATE_AUDIT = {
    "exact_duplicate_rows_including_target": full_duplicate_rows,
    "duplicate_operational_profile_rows": feature_duplicate_rows,
    "unique_operational_profiles": int(profile_groups.nunique()),
    "mixed_label_profiles": mixed_label_profiles,
    "rows_in_mixed_label_profiles": rows_in_mixed_profiles,
    "handling": "retain all rows; keep identical predictor profiles in one split group",
}
print(json.dumps(DUPLICATE_AUDIT, indent=2))

## 4. Exploratory analysis

Charts are descriptive and use the full dataset before modelling. They do not select a test threshold or tune a model.

In [ ]:
class_balance = (
    raw["churn"].value_counts().rename(index={0: "Retained", 1: "Churned"})
    .rename_axis("outcome").reset_index(name="customers")
)
class_balance["share_pct"] = 100 * class_balance["customers"] / len(raw)
display(class_balance)

churn_by_complaint = (
    raw.groupby("complains")["churn"].agg(customers="size", churn_rate="mean").reset_index()
)
churn_by_status = (
    raw.groupby("status")["churn"].agg(customers="size", churn_rate="mean").reset_index()
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.countplot(data=raw, x="churn", ax=axes[0, 0], color="#4C78A8")
axes[0, 0].set_title("Class balance")
axes[0, 0].set_xticks([0, 1], labels=["Retained", "Churned"])
sns.barplot(data=churn_by_complaint, x="complains", y="churn_rate", ax=axes[0, 1], color="#E45756")
axes[0, 1].set_title("Observed churn rate by complaint indicator")
axes[0, 1].set_ylabel("Churn rate")
sns.boxplot(data=raw, x="churn", y="frequency_of_use", ax=axes[1, 0], color="#59A14F")
axes[1, 0].set_title("Usage frequency by outcome")
axes[1, 0].set_xticks([0, 1], labels=["Retained", "Churned"])
sns.barplot(data=churn_by_status, x="status", y="churn_rate", ax=axes[1, 1], color="#F28E2B")
axes[1, 1].set_title("`Status` is a strong proxy-risk feature")
axes[1, 1].set_ylabel("Churn rate")
plt.tight_layout()
eda_chart_path = ARTIFACTS / "data_overview.png"
plt.savefig(eda_chart_path, dpi=160, bbox_inches="tight")
plt.show()

## 5. Protected holdout split

One fold is reserved as the final holdout and is not used for model selection, calibration fitting or threshold selection. `StratifiedGroupKFold` approximately preserves churn prevalence while preventing identical predictor profiles from crossing the boundary.

In [ ]:
holdout_splitter = StratifiedGroupKFold(
    n_splits=CFG.holdout_folds,
    shuffle=True,
    random_state=SEED,
)
train_index, test_index = next(holdout_splitter.split(X_raw, y, groups=profile_groups))

raw_train = X_raw.iloc[train_index].reset_index(drop=True)
raw_test = X_raw.iloc[test_index].reset_index(drop=True)
y_train = y.iloc[train_index].reset_index(drop=True)
y_test = y.iloc[test_index].reset_index(drop=True)
groups_train = profile_groups.iloc[train_index].reset_index(drop=True)
groups_test = profile_groups.iloc[test_index].reset_index(drop=True)

split_profile = pd.DataFrame([
    {
        "split": "training",
        "rows": len(raw_train),
        "predictor_profiles": groups_train.nunique(),
        "churners": int(y_train.sum()),
        "churn_rate": y_train.mean(),
    },
    {
        "split": "holdout",
        "rows": len(raw_test),
        "predictor_profiles": groups_test.nunique(),
        "churners": int(y_test.sum()),
        "churn_rate": y_test.mean(),
    },
])

group_overlap = len(set(groups_train).intersection(set(groups_test)))
assert group_overlap == 0
display(split_profile)
print(f"Duplicate-profile group overlap: {group_overlap}")

## 6. Feature policy and engineering

Two fields are excluded from the operational model:

- `age` and `age_group`: retained only for subgroup monitoring, avoiding age-based retention targeting.
- `status`: semantically close to active/non-active state and therefore tested as a potentially over-powerful proxy. The deployment candidate excludes it even if it improves prediction.

All engineered ratios use only first-nine-month measurements and fixed arithmetic; no full-dataset statistics are learned.

In [ ]:
MODEL_INPUT_COLUMNS = OPERATIONAL_INPUT_COLUMNS.copy()


def engineer_features(frame: pd.DataFrame, include_status: bool = False) -> pd.DataFrame:
    data = frame.copy()
    data = data.drop(columns=["age", "age_group"], errors="ignore")
    if not include_status:
        data = data.drop(columns=["status"], errors="ignore")

    call_denominator = data["frequency_of_use"].clip(lower=0) + 1.0
    subscription_denominator = data["subscription_length"].clip(lower=1)
    data["call_failure_rate"] = (
        data["call_failure"] / (data["call_failure"] + data["frequency_of_use"] + 1.0)
    )
    data["seconds_per_call"] = data["seconds_of_use"] / call_denominator
    data["sms_per_call"] = data["frequency_of_sms"] / call_denominator
    data["contact_diversity"] = data["distinct_called_numbers"] / call_denominator
    data["customer_value_per_month"] = data["customer_value"] / subscription_denominator
    data["usage_per_month"] = data["frequency_of_use"] / subscription_denominator
    return data.astype(float)


X_train = engineer_features(raw_train, include_status=False)
X_test = engineer_features(raw_test, include_status=False)
X_train_with_status = engineer_features(raw_train, include_status=True)

assert "status" not in X_train.columns
assert "age" not in X_train.columns and "age_group" not in X_train.columns
assert X_train.columns.tolist() == X_test.columns.tolist()
assert np.isfinite(X_train.to_numpy()).all()

feature_policy = pd.DataFrame({
    "feature": raw_train.columns,
    "operational_model": [column in MODEL_INPUT_COLUMNS for column in raw_train.columns],
    "reason_if_excluded": [
        "proxy sensitivity only" if column == "status"
        else "subgroup monitoring only" if column in {"age", "age_group"}
        else "included"
        for column in raw_train.columns
    ],
})
display(feature_policy)
print(f"Engineered model matrix: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")

## 7. Baseline-first grouped cross-validation

Fixed, reviewable configurations are compared on the training split. Average precision is the primary selection metric; ROC-AUC, balanced accuracy, recall, precision and Brier score provide complementary views.

In [ ]:
precision_scorer = make_scorer(precision_score, zero_division=0)
SCORING = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "balanced_accuracy": "balanced_accuracy",
    "recall": "recall",
    "precision": precision_scorer,
    "neg_brier": "neg_brier_score",
}

CANDIDATES = {
    "Prevalence baseline": DummyClassifier(strategy="prior"),
    "Logistic regression": Pipeline([
        ("scale", RobustScaler()),
        ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)),
    ]),
    "Random forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=3,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1,
    ),
    "Histogram gradient boosting": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=15,
        min_samples_leaf=20,
        l2_regularization=1.0,
        class_weight="balanced",
        random_state=SEED,
    ),
}

cv_splitter = StratifiedGroupKFold(
    n_splits=CFG.cross_validation_folds,
    shuffle=True,
    random_state=2026,
)

cv_rows = []
for model_name, estimator in CANDIDATES.items():
    scores = cross_validate(
        estimator,
        X_train,
        y_train,
        groups=groups_train,
        cv=cv_splitter,
        scoring=SCORING,
        n_jobs=-1,
        return_train_score=False,
    )
    for fold in range(CFG.cross_validation_folds):
        cv_rows.append({
            "model": model_name,
            "fold": fold + 1,
            "average_precision": scores["test_average_precision"][fold],
            "roc_auc": scores["test_roc_auc"][fold],
            "balanced_accuracy": scores["test_balanced_accuracy"][fold],
            "recall_at_0_5": scores["test_recall"][fold],
            "precision_at_0_5": scores["test_precision"][fold],
            "brier_score": -scores["test_neg_brier"][fold],
        })

cv_results = pd.DataFrame(cv_rows)
cv_summary = (
    cv_results.groupby("model")
    .agg(
        mean_average_precision=("average_precision", "mean"),
        sd_average_precision=("average_precision", "std"),
        mean_roc_auc=("roc_auc", "mean"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_recall_at_0_5=("recall_at_0_5", "mean"),
        mean_precision_at_0_5=("precision_at_0_5", "mean"),
        mean_brier_score=("brier_score", "mean"),
    )
    .sort_values("mean_average_precision", ascending=False)
    .reset_index()
)
display(cv_summary)

plt.figure(figsize=(11, 6))
sns.barplot(data=cv_summary, y="model", x="mean_average_precision", color="#4C78A8")
plt.axvline(y_train.mean(), color="#E45756", linestyle="--", label="Training churn prevalence")
plt.xlabel("Mean grouped-CV average precision")
plt.ylabel("")
plt.title("Model comparison on training data")
plt.legend()
plt.tight_layout()
model_comparison_chart_path = ARTIFACTS / "model_comparison.png"
plt.savefig(model_comparison_chart_path, dpi=160, bbox_inches="tight")
plt.show()

## 8. Select the nonlinear model and audit `Status`

The best non-baseline model is selected using training-only grouped-CV average precision. The same algorithm is then evaluated with and without `Status`. This is a sensitivity analysis, not permission to deploy the proxy.

In [ ]:
eligible_models = cv_summary.loc[cv_summary["model"] != "Prevalence baseline"]
selected_model_name = str(eligible_models.iloc[0]["model"])
selected_estimator_template = CANDIDATES[selected_model_name]

status_sensitivity_rows = []
for feature_policy_name, feature_matrix in {
    "Status excluded": X_train,
    "Status included": X_train_with_status,
}.items():
    scores = cross_validate(
        clone(selected_estimator_template),
        feature_matrix,
        y_train,
        groups=groups_train,
        cv=cv_splitter,
        scoring={"average_precision": "average_precision", "roc_auc": "roc_auc", "brier": "neg_brier_score"},
        n_jobs=-1,
    )
    status_sensitivity_rows.append({
        "feature_policy": feature_policy_name,
        "mean_average_precision": scores["test_average_precision"].mean(),
        "mean_roc_auc": scores["test_roc_auc"].mean(),
        "mean_brier_score": -scores["test_brier"].mean(),
    })

status_sensitivity = pd.DataFrame(status_sensitivity_rows)
status_ap_lift = float(
    status_sensitivity.set_index("feature_policy").loc["Status included", "mean_average_precision"]
    - status_sensitivity.set_index("feature_policy").loc["Status excluded", "mean_average_precision"]
)

display(status_sensitivity)
print(f"Selected algorithm: {selected_model_name}")
print(f"Average-precision lift from Status: {status_ap_lift:+.4f}")
print("Deployment policy: Status remains excluded because of semantic proxy risk.")

## 9. Grouped out-of-fold probability calibration

Raw tree-ensemble scores are converted into probabilities with Platt scaling. The calibrator is fitted on predictions generated for each training row by a model that did not train on that row or its duplicate profile. The base model is then fitted on all training rows.

In [ ]:
def safe_logit(probabilities: np.ndarray) -> np.ndarray:
    clipped = np.clip(np.asarray(probabilities, dtype=float), 1e-6, 1 - 1e-6)
    return logit(clipped).reshape(-1, 1)


oof_raw_probability = cross_val_predict(
    clone(selected_estimator_template),
    X_train,
    y_train,
    groups=groups_train,
    cv=cv_splitter,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

probability_calibrator = LogisticRegression(C=1e6, max_iter=2000, random_state=SEED)
probability_calibrator.fit(safe_logit(oof_raw_probability), y_train)
oof_calibrated_probability = probability_calibrator.predict_proba(
    safe_logit(oof_raw_probability)
)[:, 1]

final_base_model = clone(selected_estimator_template)
final_base_model.fit(X_train, y_train)


class PlattCalibratedModel:
    def __init__(self, base_model: Any, calibrator: Any):
        self.base_model = base_model
        self.calibrator = calibrator
        self.classes_ = np.array([0, 1])

    def predict_proba(self, features: pd.DataFrame) -> np.ndarray:
        raw_probability = self.base_model.predict_proba(features)[:, 1]
        calibrated = self.calibrator.predict_proba(safe_logit(raw_probability))[:, 1]
        return np.column_stack([1 - calibrated, calibrated])

    def predict(self, features: pd.DataFrame, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(features)[:, 1] >= threshold).astype(int)


calibrated_model = PlattCalibratedModel(final_base_model, probability_calibrator)

calibration_summary = pd.DataFrame([
    {
        "probabilities": "Raw grouped out-of-fold",
        "average_precision": average_precision_score(y_train, oof_raw_probability),
        "brier_score": brier_score_loss(y_train, oof_raw_probability),
        "log_loss": log_loss(y_train, oof_raw_probability),
    },
    {
        "probabilities": "Calibrated grouped out-of-fold",
        "average_precision": average_precision_score(y_train, oof_calibrated_probability),
        "brier_score": brier_score_loss(y_train, oof_calibrated_probability),
        "log_loss": log_loss(y_train, oof_calibrated_probability),
    },
])
display(calibration_summary)

## 10. Choose the operating threshold on training predictions

The default 0.5 threshold is not automatically a business decision. The table below calculates screening volume, precision, recall and scenario cost across thresholds using calibrated out-of-fold probabilities only.

In [ ]:
def threshold_metrics(target: pd.Series, probability: np.ndarray, threshold: float) -> dict[str, float]:
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(target, prediction, labels=[0, 1]).ravel()
    contact_count = tp + fp
    cost_units = (
        CFG.contact_cost_units * contact_count
        + CFG.missed_churn_cost_units * fn
    )
    return {
        "threshold": float(threshold),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "precision": precision_score(target, prediction, zero_division=0),
        "recall": recall_score(target, prediction, zero_division=0),
        "f1": f1_score(target, prediction, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(target, prediction),
        "alert_rate": prediction.mean(),
        "cost_units": float(cost_units),
    }


threshold_table = pd.DataFrame([
    threshold_metrics(y_train, oof_calibrated_probability, threshold)
    for threshold in np.round(np.arange(0.02, 0.81, 0.01), 2)
])
no_intervention_cost = float(CFG.missed_churn_cost_units * y_train.sum())
threshold_table["cost_reduction_vs_no_intervention"] = no_intervention_cost - threshold_table["cost_units"]

best_threshold_row = threshold_table.sort_values(
    ["cost_units", "recall"], ascending=[True, False]
).iloc[0]
decision_threshold = float(best_threshold_row["threshold"])

display(threshold_table.sort_values("cost_units").head(12))
print(f"Selected threshold: {decision_threshold:.2f}")
print(f"Training-only scenario cost: {best_threshold_row['cost_units']:,.0f} units")
print(f"No-intervention scenario cost: {no_intervention_cost:,.0f} units")

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(threshold_table["threshold"], threshold_table["cost_units"], color="#E45756", linewidth=2)
ax1.axvline(decision_threshold, color="black", linestyle="--", label=f"Selected: {decision_threshold:.2f}")
ax1.set_xlabel("Probability threshold")
ax1.set_ylabel("Scenario cost units", color="#E45756")
ax2 = ax1.twinx()
ax2.plot(threshold_table["threshold"], threshold_table["recall"], color="#4C78A8", label="Recall")
ax2.plot(threshold_table["threshold"], threshold_table["precision"], color="#59A14F", label="Precision")
ax2.set_ylabel("Metric value")
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc="center right")
plt.title("Training-only threshold trade-offs")
plt.tight_layout()
threshold_chart_path = ARTIFACTS / "threshold_tradeoffs.png"
plt.savefig(threshold_chart_path, dpi=160, bbox_inches="tight")
plt.show()

## 11. Final holdout evaluation

The holdout is scored once after the algorithm, calibration approach and threshold have been fixed. Threshold-independent ranking metrics and threshold-dependent decision metrics are both reported.

In [ ]:
test_raw_probability = final_base_model.predict_proba(X_test)[:, 1]
test_probability = calibrated_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= decision_threshold).astype(int)
default_prediction = (test_probability >= 0.5).astype(int)


def expected_calibration_error(target: pd.Series, probability: np.ndarray, bins: int = 10) -> float:
    frame = pd.DataFrame({"target": np.asarray(target), "probability": probability})
    frame["bin"] = pd.qcut(frame["probability"], q=bins, duplicates="drop")
    grouped = frame.groupby("bin", observed=True).agg(
        rows=("target", "size"),
        observed_rate=("target", "mean"),
        mean_probability=("probability", "mean"),
    )
    return float(
        (grouped["rows"] / len(frame) * (grouped["observed_rate"] - grouped["mean_probability"]).abs()).sum()
    )


def evaluation_row(label: str, prediction: np.ndarray) -> dict[str, float]:
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
    return {
        "operating_point": label,
        "threshold": decision_threshold if label == "Cost-selected threshold" else 0.5,
        "average_precision": average_precision_score(y_test, test_probability),
        "roc_auc": roc_auc_score(y_test, test_probability),
        "precision": precision_score(y_test, prediction, zero_division=0),
        "recall": recall_score(y_test, prediction, zero_division=0),
        "f1": f1_score(y_test, prediction, zero_division=0),
        "specificity": tn / (tn + fp),
        "balanced_accuracy": balanced_accuracy_score(y_test, prediction),
        "accuracy": accuracy_score(y_test, prediction),
        "alert_rate": prediction.mean(),
        "brier_score": brier_score_loss(y_test, test_probability),
        "log_loss": log_loss(y_test, test_probability),
        "expected_calibration_error": expected_calibration_error(y_test, test_probability),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
    }


test_metrics = pd.DataFrame([
    evaluation_row("Cost-selected threshold", test_prediction),
    evaluation_row("Default 0.50 threshold", default_prediction),
])
display(test_metrics)
print(classification_report(y_test, test_prediction, target_names=["Retained", "Churned"], digits=3))

precision_curve, recall_curve, _ = precision_recall_curve(y_test, test_probability)
false_positive_rate, true_positive_rate, _ = roc_curve(y_test, test_probability)
raw_observed, raw_predicted = calibration_curve(y_test, test_raw_probability, n_bins=8, strategy="quantile")
cal_observed, cal_predicted = calibration_curve(y_test, test_probability, n_bins=8, strategy="quantile")
holdout_confusion = confusion_matrix(y_test, test_prediction, labels=[0, 1])

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes[0, 0].plot(recall_curve, precision_curve, color="#4C78A8", linewidth=2)
axes[0, 0].axhline(y_test.mean(), color="grey", linestyle="--", label="Holdout prevalence")
axes[0, 0].set_xlabel("Recall")
axes[0, 0].set_ylabel("Precision")
axes[0, 0].set_title(f"Precision–recall curve (AP={average_precision_score(y_test, test_probability):.3f})")
axes[0, 0].legend()
axes[0, 1].plot(false_positive_rate, true_positive_rate, color="#59A14F", linewidth=2)
axes[0, 1].plot([0, 1], [0, 1], color="grey", linestyle="--")
axes[0, 1].set_xlabel("False-positive rate")
axes[0, 1].set_ylabel("True-positive rate")
axes[0, 1].set_title(f"ROC curve (AUC={roc_auc_score(y_test, test_probability):.3f})")
sns.heatmap(holdout_confusion, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1, 0])
axes[1, 0].set_xlabel("Predicted")
axes[1, 0].set_ylabel("Actual")
axes[1, 0].set_xticklabels(["Retained", "Churned"])
axes[1, 0].set_yticklabels(["Retained", "Churned"], rotation=0)
axes[1, 0].set_title(f"Confusion matrix at threshold {decision_threshold:.2f}")
axes[1, 1].plot(raw_predicted, raw_observed, marker="o", label="Raw model")
axes[1, 1].plot(cal_predicted, cal_observed, marker="o", label="Calibrated model")
axes[1, 1].plot([0, 1], [0, 1], color="grey", linestyle="--", label="Perfect calibration")
axes[1, 1].set_xlabel("Mean predicted probability")
axes[1, 1].set_ylabel("Observed churn rate")
axes[1, 1].set_title("Holdout calibration")
axes[1, 1].legend()
plt.tight_layout()
evaluation_chart_path = ARTIFACTS / "holdout_evaluation.png"
plt.savefig(evaluation_chart_path, dpi=160, bbox_inches="tight")
plt.show()

## 12. Bootstrap uncertainty intervals

The holdout is modest, with fewer than 100 churners. Stratified bootstrap intervals show how uncertain the reported metrics are. They quantify sampling variation; they do not correct dataset shift or guarantee future performance.

In [ ]:
rng = np.random.default_rng(SEED)
positive_indices = np.flatnonzero(y_test.to_numpy() == 1)
negative_indices = np.flatnonzero(y_test.to_numpy() == 0)
bootstrap_rows = []

for repetition in range(CFG.bootstrap_repetitions):
    sample_indices = np.concatenate([
        rng.choice(positive_indices, size=len(positive_indices), replace=True),
        rng.choice(negative_indices, size=len(negative_indices), replace=True),
    ])
    rng.shuffle(sample_indices)
    sampled_target = y_test.to_numpy()[sample_indices]
    sampled_probability = test_probability[sample_indices]
    sampled_prediction = (sampled_probability >= decision_threshold).astype(int)
    bootstrap_rows.append({
        "average_precision": average_precision_score(sampled_target, sampled_probability),
        "roc_auc": roc_auc_score(sampled_target, sampled_probability),
        "recall": recall_score(sampled_target, sampled_prediction, zero_division=0),
        "precision": precision_score(sampled_target, sampled_prediction, zero_division=0),
        "brier_score": brier_score_loss(sampled_target, sampled_probability),
    })

bootstrap_results = pd.DataFrame(bootstrap_rows)
point_metrics = test_metrics.set_index("operating_point").loc["Cost-selected threshold"]
bootstrap_intervals = pd.DataFrame([
    {
        "metric": metric,
        "point_estimate": float(point_metrics[metric]),
        "lower_95_pct": bootstrap_results[metric].quantile(0.025),
        "upper_95_pct": bootstrap_results[metric].quantile(0.975),
    }
    for metric in ["average_precision", "roc_auc", "recall", "precision", "brier_score"]
])
display(bootstrap_intervals)

## 13. Predictive feature importance

Permutation importance measures the decrease in holdout average precision when one engineered feature is shuffled. It describes model reliance, not causality or a recommended intervention.

In [ ]:
baseline_holdout_ap = average_precision_score(y_test, test_probability)
importance_rng = np.random.default_rng(SEED + 1)
importance_rows = []

for feature in X_test.columns:
    losses = []
    for _ in range(CFG.permutation_repetitions):
        permuted = X_test.copy()
        permuted[feature] = importance_rng.permutation(permuted[feature].to_numpy())
        permuted_probability = calibrated_model.predict_proba(permuted)[:, 1]
        losses.append(baseline_holdout_ap - average_precision_score(y_test, permuted_probability))
    importance_rows.append({
        "feature": feature,
        "mean_ap_decrease": float(np.mean(losses)),
        "sd_ap_decrease": float(np.std(losses, ddof=1)),
    })

feature_importance = pd.DataFrame(importance_rows).sort_values("mean_ap_decrease", ascending=False)
display(feature_importance.head(15))

plt.figure(figsize=(11, 7))
importance_plot = feature_importance.head(12).sort_values("mean_ap_decrease")
plt.barh(importance_plot["feature"], importance_plot["mean_ap_decrease"], color="#B279A2")
plt.xlabel("Mean decrease in holdout average precision")
plt.title("Permutation importance: predictive reliance, not causality")
plt.tight_layout()
importance_chart_path = ARTIFACTS / "feature_importance.png"
plt.savefig(importance_chart_path, dpi=160, bbox_inches="tight")
plt.show()

## 14. Subgroup performance review

Age is excluded from the model but retained for monitoring. Complaint and tariff-plan slices are also reported. Small slices are labelled; the notebook does not claim fairness from one dataset or equal metrics by chance.

In [ ]:
def safe_ratio(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else np.nan


slice_rows = []
for slice_name in ["age_group", "tariff_plan", "complains"]:
    values = raw_test[slice_name].to_numpy()
    for value in sorted(pd.unique(values)):
        mask = values == value
        target_slice = y_test.to_numpy()[mask]
        probability_slice = test_probability[mask]
        prediction_slice = test_prediction[mask]
        tn, fp, fn, tp = confusion_matrix(target_slice, prediction_slice, labels=[0, 1]).ravel()
        slice_rows.append({
            "slice": slice_name,
            "value": int(value),
            "rows": int(mask.sum()),
            "churners": int(target_slice.sum()),
            "observed_churn_rate": target_slice.mean(),
            "mean_predicted_probability": probability_slice.mean(),
            "alert_rate": prediction_slice.mean(),
            "recall": safe_ratio(tp, tp + fn),
            "false_positive_rate": safe_ratio(fp, fp + tn),
            "precision": safe_ratio(tp, tp + fp),
            "roc_auc": roc_auc_score(target_slice, probability_slice) if len(np.unique(target_slice)) == 2 else np.nan,
            "small_slice_warning": bool(mask.sum() < 50 or target_slice.sum() < 10),
        })

subgroup_metrics = pd.DataFrame(slice_rows)
display(subgroup_metrics)

age_audit = subgroup_metrics.loc[subgroup_metrics["slice"].eq("age_group")]
plt.figure(figsize=(10, 6))
positions = np.arange(len(age_audit))
width = 0.36
plt.bar(positions - width / 2, age_audit["observed_churn_rate"], width, label="Observed churn rate")
plt.bar(positions + width / 2, age_audit["mean_predicted_probability"], width, label="Mean predicted probability")
plt.xticks(positions, age_audit["value"])
plt.xlabel("Age group (audit only; excluded from model)")
plt.ylabel("Rate")
plt.title("Calibration review across age groups")
plt.legend()
plt.tight_layout()
subgroup_chart_path = ARTIFACTS / "age_group_audit.png"
plt.savefig(subgroup_chart_path, dpi=160, bbox_inches="tight")
plt.show()

## 15. Failure and uncertainty analysis

False positives consume retention capacity; false negatives are missed opportunities. The analysis preserves both error types and isolates scores close to the decision boundary for human review.

In [ ]:
test_predictions = pd.DataFrame({
    "actual_churn": y_test,
    "churn_probability": test_probability,
    "flag_for_review": test_prediction,
})
test_predictions["error_type"] = np.select(
    [
        (test_predictions["actual_churn"] == 1) & (test_predictions["flag_for_review"] == 1),
        (test_predictions["actual_churn"] == 0) & (test_predictions["flag_for_review"] == 0),
        (test_predictions["actual_churn"] == 0) & (test_predictions["flag_for_review"] == 1),
        (test_predictions["actual_churn"] == 1) & (test_predictions["flag_for_review"] == 0),
    ],
    ["True positive", "True negative", "False positive", "False negative"],
    default="Unclassified",
)

failure_frame = pd.concat([
    test_predictions,
    raw_test[["complains", "frequency_of_use", "frequency_of_sms", "call_failure", "customer_value"]],
], axis=1)
failure_summary = (
    failure_frame.groupby("error_type")
    .agg(
        rows=("actual_churn", "size"),
        mean_probability=("churn_probability", "mean"),
        complaint_rate=("complains", "mean"),
        median_usage=("frequency_of_use", "median"),
        median_sms=("frequency_of_sms", "median"),
        median_call_failures=("call_failure", "median"),
        median_customer_value=("customer_value", "median"),
    )
    .reset_index()
)

uncertainty_band = 0.03
near_boundary = np.abs(test_probability - decision_threshold) <= uncertainty_band
uncertainty_summary = {
    "probability_band": [max(0, decision_threshold - uncertainty_band), min(1, decision_threshold + uncertainty_band)],
    "customers_near_boundary": int(near_boundary.sum()),
    "share_near_boundary_pct": float(100 * near_boundary.mean()),
    "observed_churn_rate_near_boundary": float(y_test.to_numpy()[near_boundary].mean()) if near_boundary.any() else None,
}

display(failure_summary)
print(json.dumps(uncertainty_summary, indent=2))

## 16. Drift reference and controlled monitor test

The dataset has no deployment-time batch, so the notebook does not fabricate “current drift.” It exports training reference statistics and a Population Stability Index function, then verifies the monitor with a deliberately shifted copy of one feature.

In [ ]:
drift_reference = pd.DataFrame({
    "feature": X_train.columns,
    "mean": X_train.mean().values,
    "std": X_train.std().values,
    "minimum": X_train.min().values,
    "p05": X_train.quantile(0.05).values,
    "median": X_train.median().values,
    "p95": X_train.quantile(0.95).values,
    "maximum": X_train.max().values,
})


def population_stability_index(reference: pd.Series, current: pd.Series, bins: int = 10) -> float:
    reference_values = np.asarray(reference, dtype=float)
    current_values = np.asarray(current, dtype=float)
    edges = np.unique(np.quantile(reference_values, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    reference_counts, _ = np.histogram(reference_values, bins=edges)
    current_counts, _ = np.histogram(current_values, bins=edges)
    reference_share = np.clip(reference_counts / reference_counts.sum(), 1e-6, None)
    current_share = np.clip(current_counts / current_counts.sum(), 1e-6, None)
    return float(np.sum((current_share - reference_share) * np.log(current_share / reference_share)))


controlled_shift = X_train["seconds_of_use"] * 1.5
controlled_shift_psi = population_stability_index(X_train["seconds_of_use"], controlled_shift)
assert controlled_shift_psi > 0.01
display(drift_reference)
print(f"Controlled seconds-of-use shift PSI: {controlled_shift_psi:.3f}")

## 17. Validated inference path

The scoring function accepts only the ten operational inputs. It validates domains, recreates the fixed feature engineering and returns a calibrated probability plus a review recommendation. It does not use age or `Status`.

In [ ]:
def validate_customer_input(record: dict[str, Any]) -> None:
    missing = sorted(set(MODEL_INPUT_COLUMNS) - set(record))
    if missing:
        raise ValueError(f"Missing required fields: {missing}")
    numeric_values = {field: float(record[field]) for field in MODEL_INPUT_COLUMNS}
    if any(value < 0 for value in numeric_values.values()):
        raise ValueError("Numeric inputs must be non-negative.")
    if int(record["complains"]) not in {0, 1}:
        raise ValueError("complains must be 0 or 1.")
    if int(record["tariff_plan"]) not in {1, 2}:
        raise ValueError("tariff_plan must be 1 or 2.")
    if not 0 <= float(record["charge_amount"]) <= 9:
        raise ValueError("charge_amount must be between 0 and 9.")
    if float(record["subscription_length"]) <= 0:
        raise ValueError("subscription_length must be positive.")


def predict_customer(record: dict[str, Any]) -> dict[str, Any]:
    validate_customer_input(record)
    raw_record = pd.DataFrame([{field: record[field] for field in MODEL_INPUT_COLUMNS}])
    engineered_record = engineer_features(raw_record, include_status=False)
    engineered_record = engineered_record.reindex(columns=X_train.columns)
    probability = float(calibrated_model.predict_proba(engineered_record)[0, 1])
    flagged = probability >= decision_threshold
    return {
        "calibrated_churn_probability": round(probability, 4),
        "decision_threshold": round(decision_threshold, 4),
        "flag_for_retention_review": bool(flagged),
        "recommended_action": "Prioritise for human retention review" if flagged else "No proactive review under current policy",
        "important_note": "Prediction is risk screening, not a causal explanation.",
    }


smoke_record = raw_train[MODEL_INPUT_COLUMNS].median(numeric_only=True).to_dict()
smoke_record["complains"] = int(smoke_record["complains"])
smoke_record["tariff_plan"] = int(smoke_record["tariff_plan"])
smoke_result = predict_customer(smoke_record)
assert 0 <= smoke_result["calibrated_churn_probability"] <= 1
assert isinstance(smoke_result["flag_for_retention_review"], bool)
print(json.dumps(smoke_result, indent=2))
print("Inference smoke test passed.")

## 18. Optional interactive application

The interface is disabled by default so a full notebook run never blocks. Set `launch_app=True` in the configuration to launch it in Colab.

In [ ]:
def score_from_form(
    call_failure: float,
    complains: int,
    subscription_length: float,
    charge_amount: float,
    seconds_of_use: float,
    frequency_of_use: float,
    frequency_of_sms: float,
    distinct_called_numbers: float,
    tariff_plan: int,
    customer_value: float,
) -> str:
    result = predict_customer({
        "call_failure": call_failure,
        "complains": complains,
        "subscription_length": subscription_length,
        "charge_amount": charge_amount,
        "seconds_of_use": seconds_of_use,
        "frequency_of_use": frequency_of_use,
        "frequency_of_sms": frequency_of_sms,
        "distinct_called_numbers": distinct_called_numbers,
        "tariff_plan": tariff_plan,
        "customer_value": customer_value,
    })
    return json.dumps(result, indent=2)


try:
    import gradio as gr

    with gr.Blocks(title="Customer Churn Prediction") as app:
        gr.Markdown("# Customer Churn Prediction\nCalibrated screening for retention review; not an automatic decision.")
        with gr.Row():
            call_failure_input = gr.Number(value=5, minimum=0, label="Call failures")
            complains_input = gr.Dropdown([0, 1], value=0, label="Complaint made")
            subscription_input = gr.Number(value=30, minimum=1, label="Subscription length")
            charge_input = gr.Number(value=1, minimum=0, maximum=9, label="Charge amount band")
            seconds_input = gr.Number(value=3000, minimum=0, label="Seconds of use")
        with gr.Row():
            calls_input = gr.Number(value=50, minimum=0, label="Frequency of use")
            sms_input = gr.Number(value=20, minimum=0, label="Frequency of SMS")
            distinct_input = gr.Number(value=20, minimum=0, label="Distinct called numbers")
            tariff_input = gr.Dropdown([1, 2], value=1, label="Tariff plan")
            value_input = gr.Number(value=250, minimum=0, label="Customer value")
        score_button = gr.Button("Score customer", variant="primary")
        result_output = gr.Code(label="Screening result", language="json")
        score_button.click(
            score_from_form,
            [
                call_failure_input, complains_input, subscription_input, charge_input, seconds_input,
                calls_input, sms_input, distinct_input, tariff_input, value_input,
            ],
            result_output,
        )

    if CFG.launch_app:
        app.launch(share=True, debug=False)
    else:
        print("Application built. Set launch_app=True in ProjectConfig to launch it.")
except Exception as exc:
    print(f"Optional Gradio interface unavailable: {type(exc).__name__}. Inference smoke test still passed.")

## 19. Export reproducible artifacts and governance records

Generated data and artifact files are for runtime inspection. The GitHub repository needs only this notebook.

In [ ]:
decision_metrics = test_metrics.set_index("operating_point").loc["Cost-selected threshold"].to_dict()

model_bundle = {
    "base_model": final_base_model,
    "probability_calibrator": probability_calibrator,
    "decision_threshold": decision_threshold,
    "engineered_feature_order": X_train.columns.tolist(),
    "raw_input_columns": MODEL_INPUT_COLUMNS,
    "excluded_fields": {
        "status": "semantic proxy-risk feature",
        "age": "not used for targeting; audit only",
        "age_group": "not used for targeting; audit only",
    },
    "decision_costs": {
        "contact_cost_units": CFG.contact_cost_units,
        "missed_churn_cost_units": CFG.missed_churn_cost_units,
    },
    "source_fingerprint": SOURCE_FINGERPRINT,
}

model_path = ARTIFACTS / "churn_model_bundle.joblib"
joblib.dump(model_bundle, model_path)

data_card = {
    "dataset": "UCI Iranian Churn",
    "source": SOURCE_FINGERPRINT,
    "rows": len(raw),
    "columns": raw.columns.tolist(),
    "target": "churn: 1 churned, 0 retained",
    "observed_churn_rate": float(y.mean()),
    "chronology": "features from first 9 months; churn at end of month 12; 3-month planning gap",
    "duplicate_profile_audit": DUPLICATE_AUDIT,
    "limitations": [
        "No customer identifier, so duplicate profiles cannot be resolved to individuals.",
        "No event timestamps, so a true calendar-time validation split is impossible.",
        "No intervention assignment or outcome, so uplift and causal retention effects cannot be estimated.",
        "Historical records from one Iranian telecom may not generalise to another market or period.",
    ],
}

model_card = {
    "model": selected_model_name,
    "purpose": "prioritise customers for human retention review",
    "primary_metric": "average precision",
    "selected_threshold": decision_threshold,
    "holdout_metrics": decision_metrics,
    "bootstrap_intervals": bootstrap_intervals.to_dict(orient="records"),
    "status_proxy_sensitivity": status_sensitivity.to_dict(orient="records"),
    "decision_costs": model_bundle["decision_costs"],
    "intended_use": "screening and workload prioritisation with human review",
    "prohibited_use": [
        "automatic denial, pricing or adverse treatment",
        "claims of causal churn drivers",
        "use in another population without validation and recalibration",
    ],
    "monitoring": [
        "input schema and ranges",
        "feature PSI and missingness",
        "calibration and churn prevalence",
        "recall, false-positive rate and alert volume by reviewed subgroup",
    ],
}

data_card_path = ARTIFACTS / "data_card.json"
model_card_path = ARTIFACTS / "model_card.json"
data_card_path.write_text(json.dumps(data_card, indent=2, default=str), encoding="utf-8")
model_card_path.write_text(json.dumps(model_card, indent=2, default=str), encoding="utf-8")

CSV_EXPORTS = {
    "quality_profile.csv": quality_profile,
    "split_profile.csv": split_profile,
    "cv_fold_results.csv": cv_results,
    "cv_summary.csv": cv_summary,
    "status_sensitivity.csv": status_sensitivity,
    "calibration_summary.csv": calibration_summary,
    "threshold_analysis.csv": threshold_table,
    "holdout_metrics.csv": test_metrics,
    "bootstrap_intervals.csv": bootstrap_intervals,
    "feature_importance.csv": feature_importance,
    "subgroup_metrics.csv": subgroup_metrics,
    "failure_summary.csv": failure_summary,
    "drift_reference.csv": drift_reference,
    "holdout_predictions.csv": test_predictions,
}
for filename, frame in CSV_EXPORTS.items():
    frame.to_csv(ARTIFACTS / filename, index=False)

ARTIFACT_PATHS = [
    model_path,
    data_card_path,
    model_card_path,
    *[ARTIFACTS / filename for filename in CSV_EXPORTS],
    eda_chart_path,
    model_comparison_chart_path,
    threshold_chart_path,
    evaluation_chart_path,
    importance_chart_path,
    subgroup_chart_path,
]

manifest_rows = []
for path in sorted(ARTIFACT_PATHS, key=lambda item: item.name):
    manifest_rows.append({
        "file": path.name,
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })
artifact_manifest = pd.DataFrame(manifest_rows)
manifest_path = ARTIFACTS / "artifact_manifest.csv"
artifact_manifest.to_csv(manifest_path, index=False)
display(artifact_manifest)

## 20. Acceptance tests

The project fails loudly if core data, validation, model, decision or artifact assumptions are broken.

In [ ]:
decision_row = test_metrics.set_index("operating_point").loc["Cost-selected threshold"]
default_row = test_metrics.set_index("operating_point").loc["Default 0.50 threshold"]

assert SOURCE_FINGERPRINT["matches_reference_archive"]
assert raw.shape == (3150, 14)
assert raw.isna().sum().sum() == 0
assert full_duplicate_rows == 300
assert mixed_label_profiles > 0
assert group_overlap == 0
assert set(X_train.columns) == set(X_test.columns)
assert not {"age", "age_group", "status"}.intersection(X_train.columns)
assert selected_model_name != "Prevalence baseline"
assert cv_summary.iloc[0]["mean_average_precision"] > y_train.mean() + 0.20
assert 0 < decision_threshold < 1
assert threshold_table["cost_units"].min() < no_intervention_cost
assert test_probability.min() >= 0 and test_probability.max() <= 1
assert decision_row["average_precision"] > y_test.mean() + 0.20
assert decision_row["roc_auc"] > 0.80
assert decision_row["recall"] >= default_row["recall"]
assert decision_row["alert_rate"] < 0.50
assert bootstrap_intervals["lower_95_pct"].notna().all()
assert feature_importance["mean_ap_decrease"].notna().all()
assert len(subgroup_metrics) >= 9
assert controlled_shift_psi > 0.01
assert all(path.exists() and path.stat().st_size > 0 for path in ARTIFACT_PATHS)
assert artifact_manifest["sha256"].str.fullmatch(r"[0-9a-f]{64}").all()
assert manifest_path.exists() and manifest_path.stat().st_size > 0

reloaded_bundle = joblib.load(model_path)
assert math.isclose(reloaded_bundle["decision_threshold"], decision_threshold)
assert reloaded_bundle["engineered_feature_order"] == X_train.columns.tolist()

print("ALL PROJECT 03 ACCEPTANCE TESTS PASSED")

## 21. Executive findings generated from the verified run

These statements are created from the current execution rather than typed in advance.

In [ ]:
decision_row = test_metrics.set_index("operating_point").loc["Cost-selected threshold"]
top_feature = str(feature_importance.iloc[0]["feature"])

EXECUTIVE_FINDINGS = [
    f"The source contains {len(raw):,} customers with an observed churn rate of {100 * y.mean():.2f}%.",
    f"Grouped validation selected {selected_model_name}; duplicate predictor profiles never crossed the holdout boundary.",
    f"The status-excluded model achieved holdout average precision {decision_row['average_precision']:.3f} "
    f"and ROC-AUC {decision_row['roc_auc']:.3f}.",
    f"At the training-selected threshold of {decision_threshold:.2f}, holdout recall was "
    f"{100 * decision_row['recall']:.1f}% and precision was {100 * decision_row['precision']:.1f}%.",
    f"The policy flagged {100 * decision_row['alert_rate']:.1f}% of holdout customers for review.",
    f"The model relied most on {top_feature} by holdout permutation importance; this is predictive, not causal.",
    f"Including Status changed grouped-CV average precision by {status_ap_lift:+.3f}; it remained excluded because of proxy risk.",
]

for finding in EXECUTIVE_FINDINGS:
    print("•", finding)

## Interview explanation

**Problem:** Predict churn early enough for a retention team to act, while class imbalance and duplicate customer profiles can make headline accuracy misleading.

**Decision:** I used grouped stratified validation, selected by average precision, excluded age and a semantically risky `Status` proxy, calibrated out-of-fold probabilities and chose the threshold from explicit decision costs.

**Technical trade-off:** Excluding `Status` may sacrifice a small amount of predictive performance, but it makes the model easier to defend as an early-warning system instead of a restatement of whether someone is already inactive.

**Failure mode:** A random row split can place identical profiles in both train and test. That inflates performance, so identical predictor hashes are kept within one group throughout splitting and cross-validation.

**Next production step:** Validate on a later calendar cohort, estimate intervention uplift through a randomised retention experiment, replace cost assumptions with finance-approved values and monitor probability calibration, workload and subgroup error rates.

## CV bullet generated from verified metrics

Use this only after the acceptance tests pass.

In [ ]:
cv_bullet = (
    f"Built a leakage-aware telecom churn screening system on {len(raw):,} real customer records, "
    f"using duplicate-profile grouped validation, calibrated {selected_model_name.lower()} and cost-based thresholding; "
    f"achieved {decision_row['average_precision']:.3f} holdout PR-AUC and "
    f"{100 * decision_row['recall']:.1f}% recall at {100 * decision_row['precision']:.1f}% precision, "
    f"with proxy-feature, subgroup and drift-monitoring checks."
)
print(cv_bullet)

## Final checklist

- [x] Real, attributed public dataset with a three-month planning gap
- [x] Explicit data contract and duplicate-profile audit
- [x] Untouched grouped holdout and grouped cross-validation
- [x] Prevalence baseline plus interpretable and nonlinear models
- [x] Age excluded from targeting and `Status` proxy sensitivity tested
- [x] Out-of-fold probability calibration
- [x] Training-only cost-based threshold selection
- [x] PR-AUC, ROC-AUC, calibration, bootstrap intervals and failure analysis
- [x] Subgroup review, drift reference and validated inference path
- [x] Model/data cards, serialized bundle, tests and artifact manifest

**Portfolio rule:** upload this single notebook to the repository root. Do not upload the source data or generated artifact directory.